# 📚 Book Ratings Insights

What makes a book loved by readers? This notebook explores the
[goodbooks-10k](https://github.com/zygmuntz/goodbooks-10k) dataset, with 10,000
popular books from Goodreads and about 6 million ratings given by their readers.

All the logic lives in the `bookinsights` package: this notebook only calls its
functions and shows the results.

1. **Setup**: import the package
2. **The data**: download and load the files
3. **Cleaning**: genres, publication eras, authors and series

### Research questions

**Books**
1. Are some genres consistently rated higher than others?
2. Are older books rated differently than recent ones?
3. Is popularity the same as appreciation? Do the most read books get higher ratings?

**Authors and series**

4. How do the ratings of an author evolve across their works?
5. What happens as a series goes on? Are later books rated higher or lower than the first one?

**Readers**

6. How do readers rate? Are users who rate many books harsher, or more consistent, than occasional readers?

## 1. Setup

The notebook lives in the `notebooks/` folder, while the `bookinsights` package is
one level up, in the project folder. Python only looks for packages in a few places,
so we add the project folder to that list (`sys.path`).

The `autoreload` extension reloads the modules of the package whenever they change,
without restarting the kernel.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

# Jupyter runs the notebook from the folder that contains it (notebooks/),
# so the project folder is its parent
PROJECT_ROOT = os.path.dirname(os.getcwd())

# Let Python find the bookinsights package, which lives in the project folder
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [3]:
from bookinsights.cleaning import clean_books
from bookinsights.data import (
    download_data,
    load_book_tags,
    load_books,
    load_ratings,
    load_tags,
)

## 2. The data

The data files are not stored in the repository, since they are too large for
GitHub. `download_data()` downloads them into the `data/` folder the first time,
and does nothing when they are already there.

In [4]:
paths = download_data()

# relpath shows the paths starting from the project folder, which is shorter to read
{name: os.path.relpath(path, PROJECT_ROOT) for name, path in paths.items()}

{'books.csv': 'data/books.csv',
 'ratings.csv': 'data/ratings.csv',
 'tags.csv': 'data/tags.csv',
 'book_tags.csv': 'data/book_tags.csv'}

The dataset is made of four tables:

- `books`: the metadata of each book (authors, title, year, ratings);
- `ratings`: one row for each rating, from 1 to 5, given by a user to a book;
- `tags`: the names of the tags, i.e. the shelves where users put the books;
- `book_tags`: how many users put each book on each shelf.

Ratings are linked to books through `book_id`, while tags are linked through
`goodreads_book_id`.

In [5]:
books = load_books()
ratings = load_ratings()
tags = load_tags()
book_tags = load_book_tags()

for name, table in [
    ("books", books),
    ("ratings", ratings),
    ("tags", tags),
    ("book_tags", book_tags),
]:
    print(f"{name:<10} {table.shape[0]:>9,} rows  {table.shape[1]:>2} columns")

books         10,000 rows  15 columns
ratings    5,976,479 rows   3 columns
tags          34,252 rows   2 columns
book_tags    999,912 rows   3 columns


In [6]:
books.head()

,book_id,goodreads_book_id,authors,original_publication_year,title,language_code,average_rating,ratings_count,work_ratings_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url
0,1,2767052,Suzanne Collins,2008.0,"The Hunger Games (The Hunger Games, #1)",eng,4.34,4780653,4942365,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...
1,2,3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Sorcerer's Stone (Harry P...,eng,4.44,4602479,4800065,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...
2,3,41865,Stephenie Meyer,2005.0,"Twilight (Twilight, #1)",en-US,3.57,3866839,3916824,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...
3,4,2657,Harper Lee,1960.0,To Kill a Mockingbird,eng,4.25,3198671,3340896,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...
4,5,4671,F. Scott Fitzgerald,1925.0,The Great Gatsby,eng,3.89,2683664,2773745,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...


`ratings` is the largest table, with about 6 million rows. Its values are small
integers (user ids up to 53,424, book ids up to 10,000 and ratings from 1 to 5), so
`load_ratings` stores them with compact types instead of the default `int64`.

In [7]:
print(ratings.dtypes)
print(f"Memory: {ratings.memory_usage(deep=True).sum() / 1e6:.1f} MB")

user_id    int32
book_id    int16
rating      int8
dtype: object
Memory: 41.8 MB


## 3. Cleaning

`clean_books` builds the table used in the analysis. The main choices are:

- **Genres.** Tags are user shelves, and most of them are not genres (`to-read`,
  `favorites`, `owned`, ...). We keep a list of 18 genres, merging synonyms
  (e.g. `sci-fi` and `science-fiction`), and give each book its **primary genre**:
  the genre tag used by the largest number of users. `young-adult` and `children`
  describe an audience, but they are kept, since Goodreads uses them as genres.
  Rows with a negative count (6) and duplicated rows (8) are removed from the tags.
- **Publication eras.** Years are grouped into six eras; ancient books (with
  negative years, e.g. *The Epic of Gilgamesh*) fall into "before 1800".
- **Main author.** When a book lists several names, the first one is the author:
  the others are usually illustrators, translators or narrators.
- **Series.** The series and the position in it are extracted from the title, e.g.
  *(The Hunger Games, #1)*. Box sets (e.g. *#1-7*) are not single books, so they
  are not assigned to a series.

In [8]:
clean = clean_books(books, book_tags, tags)
clean[
    [
        "title",
        "main_author",
        "publication_year",
        "era",
        "series",
        "series_number",
        "genre",
    ]
].head(10)

,title,main_author,publication_year,era,series,series_number,genre
0,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,2008,2000-2009,The Hunger Games,1.0,young-adult
1,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling,1997,1950-1999,Harry Potter,1.0,fantasy
2,"Twilight (Twilight, #1)",Stephenie Meyer,2005,2000-2009,Twilight,1.0,young-adult
3,To Kill a Mockingbird,Harper Lee,1960,1950-1999,NaN,NaN,classics
4,The Great Gatsby,F. Scott Fitzgerald,1925,1900-1949,NaN,NaN,classics
5,The Fault in Our Stars,John Green,2012,2010 or later,NaN,NaN,young-adult
6,The Hobbit,J.R.R. Tolkien,1937,1900-1949,NaN,NaN,fantasy
7,The Catcher in the Rye,J.D. Salinger,1951,1950-1999,NaN,NaN,classics
8,"Angels & Demons (Robert Langdon, #1)",Dan Brown,2000,2000-2009,Robert Langdon,1.0,mystery
9,Pride and Prejudice,Jane Austen,1813,1800-1899,NaN,NaN,classics


How many books fall into each genre? A few books (cookbooks, religious books, ...)
have no tag of the list, so they have no genre.

In [9]:
clean["genre"].value_counts(dropna=False)

genre
fantasy               1230
young-adult           1116
romance               1086
mystery               1060
historical-fiction     796
classics               767
biography              584
children               555
science-fiction        476
comics                 394
thriller               306
self-help              298
history                288
horror                 260
humor                  231
philosophy             222
literary-fiction       222
poetry                  64
NaN                     45
Name: count, dtype: int64

Most books were published after 1950: older books are only the ones that are still
read today.

In [10]:
clean["era"].value_counts(sort=False, dropna=False)

era
before 1800       125
1800-1899         254
1900-1949         466
1950-1999        2946
2000-2009        3121
2010 or later    3067
NaN                21
Name: count, dtype: int64

The series with the largest number of books in the dataset:

In [11]:
clean["series"].value_counts().head(10)

series
In Death                       41
Discworld                      39
Stephanie Plum                 27
Hercule Poirot                 25
Kinsey Millhone                24
Alex Cross                     23
Jack Reacher                   23
Kay Scarpetta                  23
Anita Blake, Vampire Hunter    23
Harry Bosch                    21
Name: count, dtype: int64